# Concordance Evaluator (Notebook)

Visual-only evaluator run inside the notebook (no CLI calls):
- correction field maps and sampled correction magnitude
- rematch metrics before/after for both signs
- fixed-pair metrics before/after for both signs
- residual scatter/CDF/vector plots for the chosen sign


In [ ]:
from pathlib import Path
import importlib.util
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.coordinates import SkyCoord, search_around_sky
from astropy import units as u

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['image.origin'] = 'lower'
plt.rcParams['axes.grid'] = False


In [ ]:
# Paths + controls
cwd = Path.cwd().resolve()
if (cwd / 'models/astrometry/evaluate_catalog_astrometry.py').exists():
    REPO_ROOT = cwd
elif (cwd / 'evaluate_catalog_astrometry.py').exists() and cwd.name == 'astrometry':
    REPO_ROOT = cwd.parents[1]
else:
    REPO_ROOT = Path('/home/shemmati/JAISP')

default_fits = [
    REPO_ROOT / 'concordance_ecdfs_50ep.fits',
    REPO_ROOT / 'concordance_ecdfs.fits',
]
FITS_PATH = next((pp for pp in default_fits if pp.exists()), default_fits[-1])

TILE_ID = 'tile_x00000_y00000'
RUBIN_BAND = 'r'
EUCLID_BAND = 'VIS'
EVAL_SIGNS = [+1.0, -1.0]
PLOT_SIGN = None  # set to +1.0 or -1.0 to override auto-best
MAX_SEP_ARCSEC = 0.1
CLIP_SIGMA = 3.5

DETECT_NSIGMA = 10.0
DETECT_TOP = 5000
DETECT_LOCAL_BOX = 128
DETECT_BORDER = 10
DETECT_REFINE_R = 5


In [ ]:
# Evaluator logic in-notebook
eval_py = REPO_ROOT / 'models/astrometry/evaluate_catalog_astrometry.py'
spec = importlib.util.spec_from_file_location('eval_astrom', eval_py)
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)

with fits.open(FITS_PATH) as hdul:
    dra_name = f'{TILE_ID}.{RUBIN_BAND}.DRA'
    dde_name = f'{TILE_ID}.{RUBIN_BAND}.DDE'
    dra_map, dra_hdr = ev._get_hdu_data_by_name(hdul, dra_name)
    dde_map, _ = ev._get_hdu_data_by_name(hdul, dde_name)

rubin_tile = REPO_ROOT / f'data/rubin_tiles_ecdfs/{TILE_ID}.npz'
euclid_tile = REPO_ROOT / f'data/euclid_tiles_ecdfs/{TILE_ID}_euclid.npz'
rubin_img, rubin_wcs = ev._load_rubin_tile_for_autocat(str(rubin_tile), RUBIN_BAND)
eu_img, eu_wcs = ev._load_euclid_tile_for_autocat(str(euclid_tile), EUCLID_BAND)

ref_sky, _, _ = ev._detect_sources_to_sky(
    img=eu_img,
    wcs=eu_wcs,
    nsigma=DETECT_NSIGMA,
    top=DETECT_TOP,
    local_box=DETECT_LOCAL_BOX,
    border=DETECT_BORDER,
    refine_r=DETECT_REFINE_R,
)
cand_sky, _, _ = ev._detect_sources_to_sky(
    img=rubin_img,
    wcs=rubin_wcs,
    nsigma=DETECT_NSIGMA,
    top=DETECT_TOP,
    local_box=DETECT_LOCAL_BOX,
    border=DETECT_BORDER,
    refine_r=DETECT_REFINE_R,
)

cand_x_vis, cand_y_vis = eu_wcs.world_to_pixel(cand_sky)
map_h, map_w = dra_map.shape
dstep = int(dra_hdr.get('DSTEP', 8))
x_map, y_map = ev._map_xy_from_vis_xy(
    x_vis=np.asarray(cand_x_vis, dtype=float),
    y_vis=np.asarray(cand_y_vis, dtype=float),
    map_w=map_w,
    map_h=map_h,
    full_w=map_w * dstep,
    full_h=map_h * dstep,
    xy_origin=0,
)
dra_s = ev._bilinear_sample(dra_map, x_map, y_map)
dde_s = ev._bilinear_sample(dde_map, x_map, y_map)
valid = np.isfinite(dra_s) & np.isfinite(dde_s)
corr_mag_mas = 1000.0 * np.hypot(dra_s[valid], dde_s[valid])

ref_sky_eval = ref_sky
cand_before_sky = cand_sky[valid]
cand_x = np.asarray(cand_x_vis, dtype=float)[valid]
cand_y = np.asarray(cand_y_vis, dtype=float)[valid]


def greedy_match(ref_sky_local, cand_sky_local, max_sep_arcsec):
    idx_ref, idx_cand, sep2d, _ = search_around_sky(ref_sky_local, cand_sky_local, max_sep_arcsec * u.arcsec)
    if len(idx_ref) == 0:
        return np.array([], int), np.array([], int), np.array([], float)
    sep = sep2d.to(u.arcsec).value
    order = np.argsort(sep)
    used_r = np.zeros(len(ref_sky_local), dtype=bool)
    used_c = np.zeros(len(cand_sky_local), dtype=bool)
    ri, ci, so = [], [], []
    for k in order:
        r = int(idx_ref[k])
        c = int(idx_cand[k])
        if used_r[r] or used_c[c]:
            continue
        used_r[r] = True
        used_c[c] = True
        ri.append(r)
        ci.append(c)
        so.append(float(sep[k]))
    return np.asarray(ri, int), np.asarray(ci, int), np.asarray(so, float)


def offsets_mas(ref_sub, cand_sub):
    cosd = np.cos(np.deg2rad(ref_sub.dec.deg))
    dra = (cand_sub.ra.deg - ref_sub.ra.deg) * 3600.0 * 1000.0 * cosd
    ddec = (cand_sub.dec.deg - ref_sub.dec.deg) * 3600.0 * 1000.0
    return np.asarray(dra, float), np.asarray(ddec, float)


def robust_keep(dra, ddec, clip_sigma):
    if dra.size == 0:
        return np.zeros(0, dtype=bool)
    mra = float(np.median(dra))
    mde = float(np.median(ddec))
    sra = float(1.4826 * np.median(np.abs(dra - mra)))
    sde = float(1.4826 * np.median(np.abs(ddec - mde)))
    if not np.isfinite(sra) or sra <= 0:
        sra = float(np.std(dra) + 1e-9)
    if not np.isfinite(sde) or sde <= 0:
        sde = float(np.std(ddec) + 1e-9)
    return (np.abs(dra - mra) < clip_sigma * sra) & (np.abs(ddec - mde) < clip_sigma * sde)


def summarize(dra, ddec):
    if dra.size == 0:
        return {'n': 0, 'median': np.nan, 'p68c': np.nan, 'rms': np.nan}
    r = np.hypot(dra, ddec)
    mdra = float(np.median(dra))
    mdde = float(np.median(ddec))
    rc = np.hypot(dra - mdra, ddec - mdde)
    return {
        'n': int(dra.size),
        'median': float(np.hypot(mdra, mdde)),
        'p68c': float(np.quantile(rc, 0.68)),
        'rms': float(np.sqrt(np.mean(r * r))),
    }


def eval_rematch(cand_after_sky):
    ri, ci, sep = greedy_match(ref_sky_eval, cand_after_sky, MAX_SEP_ARCSEC)
    if len(ri) == 0:
        z = summarize(np.array([]), np.array([]))
        return {'ri': ri, 'ci': ci, 'sep': sep, 'dra': np.array([]), 'ddec': np.array([]), 'r': np.array([]), 'all': z, 'clipped': z}
    dra, ddec = offsets_mas(ref_sky_eval[ri], cand_after_sky[ci])
    keep = robust_keep(dra, ddec, CLIP_SIGMA)
    return {
        'ri': ri,
        'ci': ci,
        'sep': sep,
        'dra': dra,
        'ddec': ddec,
        'r': np.hypot(dra, ddec),
        'all': summarize(dra, ddec),
        'clipped': summarize(dra[keep], ddec[keep]),
    }


def eval_fixed_pairs(cand_after_sky):
    ri, ci, sep = greedy_match(ref_sky_eval, cand_before_sky, MAX_SEP_ARCSEC)
    if len(ri) == 0:
        z = summarize(np.array([]), np.array([]))
        return {'ri': ri, 'ci': ci, 'sep': sep, 'before_all': z, 'before_clip': z, 'after_all': z, 'after_clip': z, 'delta_clip': {'median': np.nan, 'p68c': np.nan, 'rms': np.nan}}
    dra_b, ddec_b = offsets_mas(ref_sky_eval[ri], cand_before_sky[ci])
    dra_a, ddec_a = offsets_mas(ref_sky_eval[ri], cand_after_sky[ci])
    keep = robust_keep(dra_b, ddec_b, CLIP_SIGMA)
    b_all = summarize(dra_b, ddec_b)
    b_clip = summarize(dra_b[keep], ddec_b[keep])
    a_all = summarize(dra_a, ddec_a)
    a_clip = summarize(dra_a[keep], ddec_a[keep])
    return {
        'ri': ri,
        'ci': ci,
        'sep': sep,
        'before_all': b_all,
        'before_clip': b_clip,
        'after_all': a_all,
        'after_clip': a_clip,
        'delta_clip': {
            'median': a_clip['median'] - b_clip['median'],
            'p68c': a_clip['p68c'] - b_clip['p68c'],
            'rms': a_clip['rms'] - b_clip['rms'],
        },
    }


baseline = eval_rematch(cand_before_sky)
results_by_sign = {}
for s in EVAL_SIGNS:
    ra_corr, dec_corr = ev._apply_concordance_to_catalog(
        cand_ra_deg=np.asarray(cand_before_sky.ra.deg, float),
        cand_dec_deg=np.asarray(cand_before_sky.dec.deg, float),
        dra_arcsec=dra_s[valid],
        ddec_arcsec=dde_s[valid],
        apply_sign=float(s),
    )
    cand_after = SkyCoord(ra=ra_corr * u.deg, dec=dec_corr * u.deg, frame='icrs')
    results_by_sign[float(s)] = {
        'after_sky': cand_after,
        'rematch': eval_rematch(cand_after),
        'fixed': eval_fixed_pairs(cand_after),
    }


def sign_score(sign):
    r = results_by_sign[float(sign)]
    f = r['fixed']['after_clip']
    rm = r['rematch']['clipped']
    nfix = r['fixed']['before_clip']['n']
    return (
        float(f['p68c']),
        float(f['median']),
        -int(nfix),
        float(rm['p68c']),
        float(rm['median']),
        -int(r['rematch']['all']['n']),
    )


BEST_SIGN = min([float(s) for s in EVAL_SIGNS], key=sign_score)
ACTIVE_SIGN = float(PLOT_SIGN) if PLOT_SIGN is not None else BEST_SIGN
active = results_by_sign[ACTIVE_SIGN]

dra_b = baseline['dra']
ddec_b = baseline['ddec']
r_b = baseline['r']
ci_b = baseline['ci']
dra_a = active['rematch']['dra']
ddec_a = active['rematch']['ddec']
r_a = active['rematch']['r']
ci_a = active['rematch']['ci']


In [ ]:
# Sign comparison (visual summary)
rows = []
for s in sorted(results_by_sign.keys()):
    rem = results_by_sign[s]['rematch']
    fix = results_by_sign[s]['fixed']
    rows.append([
        f'{s:+.0f}',
        f"{rem['all']['n']}",
        f"{rem['all']['median']:.2f}",
        f"{rem['clipped']['p68c']:.2f}",
        f"{fix['before_clip']['n']}",
        f"{fix['after_clip']['median']:.2f}",
        f"{fix['after_clip']['p68c']:.2f}",
        f"{fix['delta_clip']['p68c']:+.2f}",
    ])

col_labels = [
    'sign',
    'rematch n',
    'rematch median',
    'rematch p68c',
    'fixed n',
    'fixed after median',
    'fixed after p68c',
    'fixed Δp68c',
]
fig, ax = plt.subplots(figsize=(12, 2.2 + 0.45 * len(rows)))
ax.axis('off')
tbl = ax.table(cellText=rows, colLabels=col_labels, loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.0, 1.3)
ax.set_title(f'Sign sweep | best={BEST_SIGN:+.0f} | active={ACTIVE_SIGN:+.0f}', pad=10)
plt.tight_layout()
plt.show()


In [ ]:
# Correction field + sampled correction magnitude
mag_map = np.hypot(dra_map, dde_map)
q = float(np.nanpercentile(np.abs(np.concatenate([dra_map.ravel(), dde_map.ravel()])), 99))
q = max(q, 1e-6)

fig, ax = plt.subplots(2, 2, figsize=(13, 10))
im0 = ax[0, 0].imshow(dra_map, cmap='RdBu_r', vmin=-q, vmax=q)
ax[0, 0].set_title('DRA map [arcsec]')
plt.colorbar(im0, ax=ax[0, 0], fraction=0.046)
im1 = ax[0, 1].imshow(dde_map, cmap='RdBu_r', vmin=-q, vmax=q)
ax[0, 1].set_title('DDE map [arcsec]')
plt.colorbar(im1, ax=ax[0, 1], fraction=0.046)
im2 = ax[1, 0].imshow(mag_map, cmap='magma')
ax[1, 0].set_title('|delta| map [arcsec]')
plt.colorbar(im2, ax=ax[1, 0], fraction=0.046)

bins = np.linspace(0.0, max(5.0, float(np.nanpercentile(corr_mag_mas, 99) * 1.15)), 40)
ax[1, 1].hist(corr_mag_mas, bins=bins, color='tab:blue', alpha=0.85)
if corr_mag_mas.size:
    med = float(np.median(corr_mag_mas))
    p95 = float(np.quantile(corr_mag_mas, 0.95))
    ax[1, 1].axvline(med, color='black', ls='--', lw=1.5, label=f'median={med:.1f} mas')
    ax[1, 1].axvline(p95, color='crimson', ls='--', lw=1.5, label=f'p95={p95:.1f} mas')
    ax[1, 1].legend(frameon=False)
ax[1, 1].set_title('Sampled correction magnitude on candidates [mas]')
ax[1, 1].set_xlabel('mas')
ax[1, 1].set_ylabel('count')

fig.suptitle(f'{TILE_ID} | {RUBIN_BAND}->{EUCLID_BAND} | active sign={ACTIVE_SIGN:+.0f}', y=0.98)
plt.tight_layout()
plt.show()


In [ ]:
# Matched residuals before vs after (active sign)
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
lim = 1.1 * max(
    20.0,
    float(np.nanmax(np.abs(dra_b))) if dra_b.size else 0.0,
    float(np.nanmax(np.abs(ddec_b))) if ddec_b.size else 0.0,
    float(np.nanmax(np.abs(dra_a))) if dra_a.size else 0.0,
    float(np.nanmax(np.abs(ddec_a))) if ddec_a.size else 0.0,
)

b = baseline['clipped']
a = active['rematch']['clipped']
ax[0].scatter(dra_b, ddec_b, s=18, alpha=0.8, color='tab:orange')
ax[0].axhline(0, color='0.7', lw=1)
ax[0].axvline(0, color='0.7', lw=1)
ax[0].set_title(f"Before (n={baseline['all']['n']})\nmedian={b['median']:.1f} mas, p68c={b['p68c']:.1f} mas")
ax[0].set_xlabel('dRA* [mas]')
ax[0].set_ylabel('dDec [mas]')
ax[0].set_xlim(-lim, lim)
ax[0].set_ylim(-lim, lim)
ax[0].set_aspect('equal', 'box')

ax[1].scatter(dra_a, ddec_a, s=18, alpha=0.8, color='tab:green')
ax[1].axhline(0, color='0.7', lw=1)
ax[1].axvline(0, color='0.7', lw=1)
ax[1].set_title(f"After (n={active['rematch']['all']['n']})\nmedian={a['median']:.1f} mas, p68c={a['p68c']:.1f} mas")
ax[1].set_xlabel('dRA* [mas]')
ax[1].set_ylabel('dDec [mas]')
ax[1].set_xlim(-lim, lim)
ax[1].set_ylim(-lim, lim)
ax[1].set_aspect('equal', 'box')

if r_b.size:
    xb = np.sort(r_b)
    yb = np.arange(1, len(xb) + 1) / len(xb)
    ax[2].plot(xb, yb, lw=2, color='tab:orange', label='before')
if r_a.size:
    xa = np.sort(r_a)
    ya = np.arange(1, len(xa) + 1) / len(xa)
    ax[2].plot(xa, ya, lw=2, color='tab:green', label='after')
ax[2].set_title('Radial residual CDF (rematch)')
ax[2].set_xlabel('offset [mas]')
ax[2].set_ylabel('CDF')
ax[2].set_ylim(0, 1.0)
ax[2].legend(frameon=False)

plt.tight_layout()
plt.show()


In [ ]:
# Residual vectors in VIS pixel space (rematch)
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

if len(ci_b):
    ax[0].quiver(cand_x[ci_b], cand_y[ci_b], dra_b, ddec_b, angles='xy', scale_units='xy', scale=250, color='tab:orange', alpha=0.85)
ax[0].set_title(f'Before vectors (n={len(ci_b)})')
ax[0].set_xlabel('x_VIS [pix]')
ax[0].set_ylabel('y_VIS [pix]')
ax[0].invert_yaxis()

if len(ci_a):
    ax[1].quiver(cand_x[ci_a], cand_y[ci_a], dra_a, ddec_a, angles='xy', scale_units='xy', scale=250, color='tab:green', alpha=0.85)
ax[1].set_title(f'After vectors (n={len(ci_a)}) | sign={ACTIVE_SIGN:+.0f}')
ax[1].set_xlabel('x_VIS [pix]')
ax[1].set_ylabel('y_VIS [pix]')
ax[1].invert_yaxis()

plt.tight_layout()
plt.show()
